In [8]:
from pybaseball import bwar_bat, bwar_pitch, cache

cache.enable()

bat = bwar_bat()
print(bat.shape)
print(bat.columns.tolist())
bat[bat["year_ID"] == 2023].head()

(126547, 17)
['name_common', 'mlb_ID', 'player_ID', 'year_ID', 'team_ID', 'stint_ID', 'lg_ID', 'pitcher', 'G', 'PA', 'salary', 'runs_above_avg', 'runs_above_avg_off', 'runs_above_avg_def', 'WAR_rep', 'WAA', 'WAR']


,name_common,mlb_ID,player_ID,year_ID,team_ID,stint_ID,lg_ID,pitcher,G,PA,salary,runs_above_avg,runs_above_avg_off,runs_above_avg_def,WAR_rep,WAA,WAR
66,Fernando Abad,472551.0,abadfe01,2023,COL,1,NL,Y,0,0,100000.0,0.0,0.0,0.0,NaN,NaN,NaN
90,Andrew Abbott,671096.0,abbotan01,2023,CIN,1,NL,Y,0,0,NaN,0.0,0.0,0.0,NaN,NaN,NaN
95,Cory Abbott,676265.0,abbotco01,2023,WSN,1,NL,Y,0,0,NaN,0.0,0.0,0.0,NaN,NaN,NaN
226,CJ Abrams,682928.0,abramcj01,2023,WSN,1,NL,N,151,614,724200.0,14.8,10.8,12.8,1.98,1.47,3.45
237,Albert Abreu,656061.0,abreual01,2023,NYY,1,AL,Y,0,0,738700.0,0.0,0.0,0.0,NaN,NaN,NaN


In [9]:
import pandas as pd

In [3]:
print(bat["year_ID"].min(), bat["year_ID"].max())

pit = bwar_pitch()
print(pit.shape)
print(pit.columns.tolist())
pit[pit["year_ID"] == 2023].head()

1871 2026
(57884, 19)
['name_common', 'mlb_ID', 'player_ID', 'year_ID', 'team_ID', 'stint_ID', 'lg_ID', 'G', 'GS', 'RA', 'xRA', 'BIP', 'BIP_perc', 'salary', 'ERA_plus', 'WAR_rep', 'WAA', 'WAA_adj', 'WAR']


,name_common,mlb_ID,player_ID,year_ID,team_ID,stint_ID,lg_ID,G,GS,RA,xRA,BIP,BIP_perc,salary,ERA_plus,WAR_rep,WAA,WAA_adj,WAR
53911,Albert Abreu,656061.0,abreual01,2023,NYY,1,AL,45,0,39,32.313,165,0.0408,738700.0,91.967742,0.5697,-0.8145,0.0027,-0.24
53912,Bryan Abreu,650556.0,abreubr01,2023,HOU,1,AL,72,0,17,37.368,153,0.0379,745200.0,243.357143,0.6945,1.7064,0.1199,2.52
53913,Domingo Acevedo,642758.0,acevedo01,2023,OAK,1,AL,9,0,11,4.798,37,0.0086,725000.0,41.090909,0.0904,-0.5877,-0.1571,-0.65
53914,Garrett Acton,670183.0,actonga01,2023,OAK,1,AL,6,0,8,3.419,21,0.0049,NaN,35.137500,0.0549,-0.4476,0.0065,-0.39
53915,Jason Adam,592094.0,adamja01,2023,TBR,1,AL,56,0,21,27.641,124,0.0311,1775000.0,141.627778,0.5289,0.2856,-0.0905,0.72


In [4]:
bat_s = (bat[bat["PA"] > 0]
         .groupby(["mlb_ID", "year_ID"])
         .agg(name=("name_common", "first"), PA=("PA", "sum"), WAR=("WAR", "sum"))
         .reset_index())

pit_s = (pit.groupby(["mlb_ID", "year_ID"])
         .agg(name=("name_common", "first"), G=("G", "sum"), GS=("GS", "sum"),
              BIP=("BIP", "sum"), WAR=("WAR", "sum"))
         .reset_index())

print(bat_s[bat_s["year_ID"] >= 2022].groupby("year_ID")["PA"].sum())
print(bat_s[bat_s["year_ID"] == 2026].nlargest(5, "WAR")[["name", "PA", "WAR"]])

year_ID
2022    182052
2023    184104
2024    182449
2025    182926
2026    183849
Name: PA, dtype: int64
                      name   PA   WAR
89862  Pete Crow-Armstrong  726  9.83
88603         Brice Turang  680  7.34
90083      Kevin McGonigle  707  7.05
88922       Yordan Alvarez  692  6.77
89330       Bobby Witt Jr.  632  6.72


In [12]:
import pandas as pd
import unicodedata, re

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = s.replace("-", " ")
    s = re.sub(r"\b(jr|sr|ii|iii|iv)\b\.?", "", s)
    s = re.sub(r"[^a-z ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return {"michael fiers": "mike fiers", "steven pearce": "steve pearce"}.get(s, s)

bat_s["key"] = bat_s["name"].apply(norm)
pit_s["key"] = pit_s["name"].apply(norm)

c = pd.read_csv("../data/processed/contracts.csv")
c["key"] = c["player_name"].apply(norm)
c["stat_year"] = c["signing_year"] - 1

hit = c[c["player_type"] == "hitter"].merge(
    bat_s[["key", "year_ID", "mlb_ID", "WAR"]],
    left_on=["key", "stat_year"], right_on=["key", "year_ID"], how="left")
pitch = c[c["player_type"] == "pitcher"].merge(
    pit_s[["key", "year_ID", "mlb_ID", "WAR"]],
    left_on=["key", "stat_year"], right_on=["key", "year_ID"], how="left")

matched = pd.concat([hit, pitch], ignore_index=True)

print("Contracts:", len(c), "| Rows after join:", len(matched))

# Drop contracts where one name matched two different players
amb = matched.duplicated(["player_name", "signing_year"], keep=False)
print("Excluding", amb.sum(), "rows from",
      matched[amb][["player_name", "signing_year"]].drop_duplicates().shape[0],
      "ambiguous contracts")
matched = matched[~amb].copy()

print(matched.shape)
print(matched.groupby("player_type")["mlb_ID"].apply(lambda s: s.notna().mean()))

matched.to_csv("../data/processed/contracts_matched.csv", index=False)

Contracts: 1623 | Rows after join: 1628
Excluding 10 rows from 5 ambiguous contracts
(1618, 13)
player_type
hitter     0.950292
pitcher    0.887580
Name: mlb_ID, dtype: float64


In [13]:
seasons = {
    "hitter": bat_s[["mlb_ID", "year_ID", "key", "WAR"]],
    "pitcher": pit_s[["mlb_ID", "year_ID", "key", "WAR"]],
}

# 1. Fill in missing IDs by looking back up to 2 extra seasons
def find_id(row):
    if pd.notna(row["mlb_ID"]):
        return row["mlb_ID"]
    s = seasons[row["player_type"]]
    cand = s[(s["key"] == row["key"]) &
             (s["year_ID"].between(row["stat_year"] - 2, row["stat_year"]))]
    ids = cand["mlb_ID"].unique()
    return ids[0] if len(ids) == 1 else float("nan")

m = matched.copy()
m["mlb_ID"] = m.apply(find_id, axis=1)
print("Share with a player ID:", round(m["mlb_ID"].notna().mean(), 3))

# 2. Look up WAR for each of the three seasons before signing
lookup = {}
for t, s in seasons.items():
    for mid, yr, w in zip(s["mlb_ID"], s["year_ID"], s["WAR"]):
        lookup[(t, mid, yr)] = w

for back, col in enumerate(["war_1", "war_2", "war_3"]):
    m[col] = [
        lookup.get((t, i, y - back), 0.0) if pd.notna(i) else float("nan")
        for t, i, y in zip(m["player_type"], m["mlb_ID"], m["stat_year"])
    ]

# 3. Weighted average (most recent season counts most)
m["war_w"] = (5 * m["war_1"] + 4 * m["war_2"] + 3 * m["war_3"]) / 12

m.to_csv("../data/processed/contracts_with_war.csv", index=False)
print(m.shape)
print(m[["war_w", "war_1", "aav"]].corr()["aav"])
m.nlargest(8, "aav")[["player_name", "signing_year", "war_3", "war_2", "war_1", "war_w", "aav"]]

Share with a player ID: 0.959
(1618, 17)
war_w    0.743842
war_1    0.701687
aav      1.000000
Name: aav, dtype: float64


,player_name,signing_year,war_3,war_2,war_1,war_w,aav
1330,Shohei Ohtani,2024,4.07,6.25,3.80,4.684167,7.000000e+07
630,Kyle Tucker,2026,5.51,4.61,4.53,4.801667,6.000000e+07
561,Juan Soto,2025,5.47,5.29,8.00,6.464167,5.100000e+07
1163,Max Scherzer,2022,5.28,2.18,6.07,4.575833,4.333333e+07
1256,Justin Verlander,2023,0.16,0.00,5.61,2.377500,4.333333e+07
635,Bo Bichette,2026,4.75,-0.34,3.44,2.507500,4.200000e+07
434,Aaron Judge,2023,1.14,5.89,10.77,6.735833,4.000000e+07
563,Alex Bregman,2025,4.49,4.87,3.98,4.404167,4.000000e+07


In [14]:
from pybaseball import chadwick_register

reg = chadwick_register()
print(reg.shape)
print([c for c in reg.columns if "birth" in c.lower() or "mlbam" in c.lower()])
reg.head(3)

Gathering player lookup table. This may take a moment.
(26138, 8)
['key_mlbam']


,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last
0,Bradley,Jed,605152,bradj002,bradlje01,13166,2016.0,2016.0
1,Barrios,Manuel,110625,barrm002,barrima01,1000605,1997.0,1998.0
2,Martin,Frank,118336,martf102,martifr01,1008165,1897.0,1899.0


In [15]:
import requests, time

ids = sorted(int(i) for i in m["mlb_ID"].dropna().unique())
print("Players to look up:", len(ids))

births = {}
for i in range(0, len(ids), 100):
    batch = ids[i:i+100]
    r = requests.get(
        "https://statsapi.mlb.com/api/v1/people",
        params={"personIds": ",".join(map(str, batch))},
        timeout=30,
    )
    r.raise_for_status()
    for p in r.json().get("people", []):
        if "birthDate" in p:
            births[p["id"]] = p["birthDate"]
    time.sleep(0.5)

print("Birth dates found:", len(births))

m["birth_date"] = pd.to_datetime(m["mlb_ID"].map(births))

# Age at signing: the offseason starts after season N, so use Dec 1 of the stat year
signing_date = pd.to_datetime(m["stat_year"].astype(int).astype(str) + "-12-01")
m["age_at_signing"] = ((signing_date - m["birth_date"]).dt.days / 365.25).round(1)

print(m["age_at_signing"].describe())
m.to_csv("../data/processed/contracts_with_age.csv", index=False)
m.nlargest(5, "aav")[["player_name", "signing_year", "birth_date", "age_at_signing", "aav"]]

Players to look up: 931
Birth dates found: 931
count    1552.000000
mean       32.191559
std         3.007202
min        23.700000
25%        30.100000
50%        31.800000
75%        34.000000
max        44.100000
Name: age_at_signing, dtype: float64


,player_name,signing_year,birth_date,age_at_signing,aav
1330,Shohei Ohtani,2024,1994-07-05,29.4,7.000000e+07
630,Kyle Tucker,2026,1997-01-17,28.9,6.000000e+07
561,Juan Soto,2025,1998-10-25,26.1,5.100000e+07
1163,Max Scherzer,2022,1984-07-27,37.3,4.333333e+07
1256,Justin Verlander,2023,1983-02-20,39.8,4.333333e+07


In [16]:
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("../data/processed/contracts_with_age.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2

def run_baseline(df, label):
    X = pd.get_dummies(
        df[["war_w", "age_at_signing", "age_sq", "qualifying_offer", "position"]],
        columns=["position"], dtype=float,
    )
    train = df["signing_year"] <= 2023
    test = df["signing_year"] >= 2024

    model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    model.fit(X[train], df.loc[train, "aav_m"])

    pred = model.predict(X[test])
    actual = df.loc[test, "aav_m"]
    naive = np.full(len(actual), df.loc[train, "aav_m"].median())

    print(f"{label} | train {train.sum()} | test {test.sum()}")
    print(f"  Model MAE: ${mean_absolute_error(actual, pred):.2f}M")
    print(f"  Naive MAE (always guess the median): ${mean_absolute_error(actual, naive):.2f}M")

run_baseline(d, "All contracts")
run_baseline(d[d["total_value"] >= 5_000_000], "$5M+ contracts only")

All contracts | train 1103 | test 448
  Model MAE: $3.53M
  Naive MAE (always guess the median): $5.62M
$5M+ contracts only | train 561 | test 213
  Model MAE: $4.38M
  Naive MAE (always guess the median): $6.57M


In [17]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = pd.read_csv("../data/processed/contracts_with_age.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2

X = pd.get_dummies(
    d[["war_w", "age_at_signing", "age_sq", "qualifying_offer", "position"]],
    columns=["position"], dtype=float,
)
train = d["signing_year"] <= 2023
test = d["signing_year"] >= 2024

model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
model.fit(X[train], d.loc[train, "aav_m"])

d["pred_m"] = model.predict(X)
d["error_m"] = d["pred_m"] - d["aav_m"]   # negative = model guessed too low

print("Average error by year (in $M, negative = guessed too low):")
print(d.groupby("signing_year")["error_m"].mean().round(2))

t = d[test].copy()
t["abs_err"] = t["error_m"].abs()
print("\nBiggest misses in the test years:")
print(t.nlargest(10, "abs_err")[
    ["player_name", "signing_year", "position", "war_w", "age_at_signing", "aav_m", "pred_m"]
].round(1))

Average error by year (in $M, negative = guessed too low):
signing_year
2015    0.59
2016    0.94
2017   -0.23
2018    0.67
2019    0.58
2020    0.84
2021    0.69
2022   -1.49
2023   -2.05
2024   -0.84
2025   -1.13
2026   -1.86
Name: error_m, dtype: float64

Biggest misses in the test years:
         player_name  signing_year position  war_w  age_at_signing  aav_m  \
626      Kyle Tucker          2026       RF    4.8            28.9   60.0   
631      Bo Bichette          2026       SS    2.5            27.7   42.0   
627     Alex Bregman          2026       3B    4.1            31.7   35.0   
1417     Blake Snell          2025       SP    3.6            32.0   36.4   
629      Pete Alonso          2026       1B    3.0            31.0   31.0   
557        Juan Soto          2025       RF    6.5            26.1   51.0   
1434  Walker Buehler          2025       SP   -0.4            30.3   21.0   
628   Cody Bellinger          2026       CF    4.0            30.4   32.5   
559     Alex B

In [18]:
from sklearn.metrics import mean_absolute_error

def fit_eval(df, use_year=False, use_log=False):
    cols = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "position"]
    X = pd.get_dummies(df[cols], columns=["position"], dtype=float)
    if use_year:
        X["year"] = df["signing_year"] - 2015
    tr = df["signing_year"] <= 2023
    te = df["signing_year"] >= 2024
    y = np.log(df["aav_m"]) if use_log else df["aav_m"]
    model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    model.fit(X[tr], y[tr])
    pred = model.predict(X[te])
    if use_log:
        pred = np.exp(pred)
    actual = df.loc[te, "aav_m"].values
    return mean_absolute_error(actual, pred), (pred - actual).mean()

variants = [
    ("Baseline", False, False),
    ("+ year trend", True, False),
    ("+ year trend, log target", True, True),
]

for label, subset in [("All contracts", d), ("$5M+ only", d[d["total_value"] >= 5_000_000])]:
    print(label)
    for name, uy, ul in variants:
        mae, bias = fit_eval(subset, uy, ul)
        print(f"  {name:28s} MAE ${mae:.2f}M | avg error ${bias:+.2f}M")

All contracts
  Baseline                     MAE $3.53M | avg error $-1.28M
  + year trend                 MAE $3.74M | avg error $+0.34M
  + year trend, log target     MAE $3.91M | avg error $-1.03M
$5M+ only
  Baseline                     MAE $4.38M | avg error $-2.65M
  + year trend                 MAE $4.00M | avg error $-0.16M
  + year trend, log target     MAE $4.05M | avg error $-1.19M


In [19]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("../data/processed/contracts_with_age.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2

# Season-by-season WAR lookup from the files saved earlier
bh = pd.read_csv("../data/processed/bwar_hitters.csv")
bp = pd.read_csv("../data/processed/bwar_pitchers.csv")
war_lookup = {}
for t, s in [("hitter", bh), ("pitcher", bp)]:
    for mid, yr, w in zip(s["mlb_ID"], s["year_ID"], s["WAR"]):
        war_lookup[(t, mid, yr)] = w

# Best single season in the 5 years before signing
d["peak5"] = [
    max(war_lookup.get((t, i, y - k), 0.0) for k in range(5))
    for t, i, y in zip(d["player_type"], d["mlb_ID"], d["stat_year"])
]

def fit_eval(df, extra=()):
    cols = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "position"] + list(extra)
    X = pd.get_dummies(df[cols], columns=["position"], dtype=float)
    X["year"] = df["signing_year"] - 2015
    tr = df["signing_year"] <= 2023
    te = df["signing_year"] >= 2024
    model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    model.fit(X[tr], df.loc[tr, "aav_m"])
    pred = model.predict(X[te])
    actual = df.loc[te, "aav_m"].values
    return mean_absolute_error(actual, pred), (pred - actual).mean()

for label, subset in [("All contracts", d), ("$5M+ only", d[d["total_value"] >= 5_000_000])]:
    print(label)
    for name, extra in [("Year trend", ()), ("Year trend + peak WAR", ("peak5",))]:
        mae, bias = fit_eval(subset, extra)
        print(f"  {name:24s} MAE ${mae:.2f}M | avg error ${bias:+.2f}M")

print(d[d["player_name"] == "Walker Buehler"][["signing_year", "war_w", "peak5", "aav_m"]])

All contracts
  Year trend               MAE $3.74M | avg error $+0.34M
  Year trend + peak WAR    MAE $3.71M | avg error $+0.19M
$5M+ only
  Year trend               MAE $4.00M | avg error $-0.16M
  Year trend + peak WAR    MAE $3.99M | avg error $-0.59M
      signing_year  war_w  peak5  aav_m
1434          2025  -0.39   7.13  21.05


In [20]:
from pybaseball import statcast_batter_percentile_ranks, statcast_pitcher_percentile_ranks

sav_b = statcast_batter_percentile_ranks(2023)
print(sav_b.shape)
print(sav_b.columns.tolist())

sav_p = statcast_pitcher_percentile_ranks(2023)
print(sav_p.shape)
print(sav_p.columns.tolist())

(614, 23)
['player_name', 'player_id', 'year', 'xwoba', 'xba', 'xslg', 'xiso', 'xobp', 'brl', 'brl_percent', 'exit_velocity', 'max_ev', 'hard_hit_percent', 'k_percent', 'bb_percent', 'whiff_percent', 'chase_percent', 'arm_strength', 'sprint_speed', 'oaa', 'bat_speed', 'squared_up_rate', 'swing_length']
(707, 22)
['player_name', 'player_id', 'year', 'xwoba', 'xba', 'xslg', 'xiso', 'xobp', 'brl', 'brl_percent', 'exit_velocity', 'max_ev', 'hard_hit_percent', 'k_percent', 'bb_percent', 'whiff_percent', 'chase_percent', 'arm_strength', 'xera', 'fb_velocity', 'fb_spin', 'curve_spin']


In [21]:
import pandas as pd, time
from pybaseball import statcast_batter_percentile_ranks, statcast_pitcher_percentile_ranks, cache

cache.enable()

hit_frames, pit_frames = [], []
for yr in range(2015, 2027):
    try:
        hb = statcast_batter_percentile_ranks(yr)
        hb["year"] = yr
        pp = statcast_pitcher_percentile_ranks(yr)
        pp["year"] = yr
        hit_frames.append(hb)
        pit_frames.append(pp)
        print(yr, "hitters:", len(hb), "pitchers:", len(pp))
    except Exception as e:
        print(yr, "FAILED:", str(e)[:100])
    time.sleep(1)

sav_hit = pd.concat(hit_frames, ignore_index=True)
sav_pit = pd.concat(pit_frames, ignore_index=True)
sav_hit.to_csv("../data/processed/savant_hitters.csv", index=False)
sav_pit.to_csv("../data/processed/savant_pitchers.csv", index=False)

# Are these 0-100 percentiles?
print(sav_hit[["xwoba", "exit_velocity"]].describe().loc[["min", "max"]])

# Share of players with a value for each metric, by year
cols = ["xwoba", "brl_percent", "exit_velocity", "chase_percent", "sprint_speed", "bat_speed"]
print(sav_hit.groupby("year")[cols].apply(lambda g: g.notna().mean()).round(2))

# Does player_id match the MLB IDs we already have?
d = pd.read_csv("../data/processed/contracts_with_age.csv")
ids = set(d["mlb_ID"].dropna().astype(int))
found = ids & (set(sav_hit["player_id"].astype(int)) | set(sav_pit["player_id"].astype(int)))
print("Contract players found in Savant data:", round(len(found) / len(ids), 3))

2015 hitters: 683 pitchers: 649
2016 hitters: 693 pitchers: 657
2017 hitters: 678 pitchers: 676
2018 hitters: 678 pitchers: 695
2019 hitters: 688 pitchers: 705
2020 hitters: 507 pitchers: 585
2021 hitters: 685 pitchers: 735
2022 hitters: 628 pitchers: 704
2023 hitters: 614 pitchers: 707
2024 hitters: 606 pitchers: 696
2025 hitters: 617 pitchers: 712
2026 hitters: 612 pitchers: 706
     xwoba  exit_velocity
min    1.0            1.0
max  100.0          100.0
      xwoba  brl_percent  exit_velocity  chase_percent  sprint_speed  \
year                                                                   
2015   0.37         0.37           0.37           0.37           1.0   
2016   0.35         0.35           0.35           0.35           1.0   
2017   0.38         0.38           0.38           0.38           1.0   
2018   0.37         0.37           0.37           0.37           1.0   
2019   0.36         0.36           0.36           0.36           1.0   
2020   0.50         0.50          

In [22]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

d = pd.read_csv("../data/processed/contracts_with_age.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d = d[d["total_value"] >= 5_000_000].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2
d["year"] = d["signing_year"] - 2015

sav = {
    "hitter": pd.read_csv("../data/processed/savant_hitters.csv"),
    "pitcher": pd.read_csv("../data/processed/savant_pitchers.csv"),
}
feats = {
    "hitter": ["xwoba", "brl_percent", "exit_velocity", "chase_percent",
               "whiff_percent", "bb_percent", "sprint_speed"],
    "pitcher": ["xera", "fb_velocity", "k_percent", "bb_percent",
                "whiff_percent", "chase_percent", "hard_hit_percent"],
}

def savant_avg(df, t):
    s, f = sav[t], feats[t]
    lookup = dict(zip(zip(s["player_id"].astype(int), s["year"].astype(int)),
                      s[f].values.astype(float)))
    out = []
    for mid, sy in zip(df["mlb_ID"], df["stat_year"]):
        if pd.isna(mid):
            out.append([np.nan] * len(f))
            continue
        num, den = np.zeros(len(f)), np.zeros(len(f))
        for k, w in enumerate([5, 4, 3]):
            arr = lookup.get((int(mid), int(sy) - k))
            if arr is None:
                continue
            ok = ~np.isnan(arr)
            num[ok] += w * arr[ok]
            den[ok] += w
        out.append(np.where(den > 0, num / np.where(den > 0, den, 1), np.nan))
    return pd.DataFrame(out, columns=f, index=df.index)

def compare(t):
    df = d[d["player_type"] == t].copy()
    f = feats[t]
    total = len(df)
    df = pd.concat([df, savant_avg(df, t)], axis=1)
    df = df[df[f[0]].notna()].copy()      # require the main quality metric
    df[f] = df[f].fillna(50)              # other gaps: league-median percentile
    tr = (df["signing_year"] <= 2023).values
    te = ~tr
    print(f"\n{t.upper()}S: {len(df)} of {total} contracts have Savant data "
          f"| train {tr.sum()} | test {te.sum()}")

    def fit(cols):
        X = pd.get_dummies(df[cols + ["position"]], columns=["position"], dtype=float)
        model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
        model.fit(X[tr], df.loc[tr, "aav_m"])
        return mean_absolute_error(df.loc[te, "aav_m"], model.predict(X[te]))

    base = ["age_at_signing", "age_sq", "qualifying_offer", "year"]
    naive = mean_absolute_error(df.loc[te, "aav_m"],
                                np.full(te.sum(), df.loc[tr, "aav_m"].median()))
    print(f"  Naive (median)              MAE ${naive:.2f}M")
    print(f"  A: WAR model                MAE ${fit(base + ['war_w']):.2f}M")
    print(f"  B: WAR + Savant             MAE ${fit(base + ['war_w'] + f):.2f}M")
    print(f"  C: Savant only (no WAR)     MAE ${fit(base + f):.2f}M")

compare("hitter")
compare("pitcher")


HITTERS: 291 of 328 contracts have Savant data | train 214 | test 77
  Naive (median)              MAE $7.94M
  A: WAR model                MAE $4.93M
  B: WAR + Savant             MAE $4.18M
  C: Savant only (no WAR)     MAE $4.94M

PITCHERS: 398 of 446 contracts have Savant data | train 271 | test 127
  Naive (median)              MAE $5.96M
  A: WAR model                MAE $3.69M
  B: WAR + Savant             MAE $3.43M
  C: Savant only (no WAR)     MAE $3.87M


In [23]:
def rolling(t, test_years=range(2021, 2027)):
    df = d[d["player_type"] == t].copy()
    f = feats[t]
    df = pd.concat([df, savant_avg(df, t)], axis=1)
    df = df[df[f[0]].notna()].copy()
    df[f] = df[f].fillna(50)
    base = ["age_at_signing", "age_sq", "qualifying_offer", "year"]

    def fit_predict(cols, tr, te):
        X = pd.get_dummies(df[cols + ["position"]], columns=["position"], dtype=float)
        model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
        model.fit(X[tr], df.loc[tr, "aav_m"])
        return model.predict(X[te])

    rows = []
    for yr in test_years:
        tr = (df["signing_year"] < yr).values
        te = (df["signing_year"] == yr).values
        if te.sum() == 0 or tr.sum() < 30:
            continue
        actual = df.loc[te, "aav_m"].values
        a = fit_predict(base + ["war_w"], tr, te)
        b = fit_predict(base + ["war_w"] + f, tr, te)
        rows.append((yr, te.sum(), np.abs(a - actual).mean(), np.abs(b - actual).mean()))

    out = pd.DataFrame(rows, columns=["test_year", "n", "MAE_A_WAR", "MAE_B_WAR_Savant"])
    print(f"\n{t.upper()}S")
    print(out.round(2).to_string(index=False))
    pooled_a = (out["n"] * out["MAE_A_WAR"]).sum() / out["n"].sum()
    pooled_b = (out["n"] * out["MAE_B_WAR_Savant"]).sum() / out["n"].sum()
    print(f"Pooled MAE: A ${pooled_a:.2f}M | B ${pooled_b:.2f}M")
    print("B beats A in", (out["MAE_B_WAR_Savant"] < out["MAE_A_WAR"]).sum(), "of", len(out), "years")

rolling("hitter")
rolling("pitcher")


HITTERS
 test_year  n  MAE_A_WAR  MAE_B_WAR_Savant
      2021 27       2.98              3.08
      2022 31       3.70              3.77
      2023 38       3.12              3.19
      2024 21       3.96              3.31
      2025 29       4.14              3.35
      2026 27       6.07              5.45
Pooled MAE: A $3.93M | B $3.67M
B beats A in 3 of 6 years

PITCHERS
 test_year  n  MAE_A_WAR  MAE_B_WAR_Savant
      2021 33       2.98              2.95
      2022 48       3.20              3.05
      2023 47       3.55              3.47
      2024 41       3.41              3.32
      2025 44       4.11              3.90
      2026 42       3.38              2.92
Pooled MAE: A $3.46M | B $3.28M
B beats A in 6 of 6 years


In [26]:
import re
import pandas as pd

pos_map = [
    ("catcher", "C"), ("first base", "1B"), ("second base", "2B"),
    ("third base", "3B"), ("shortstop", "SS"),
    ("left field", "LF"), ("center field", "CF"), ("right field", "RF"),
    ("designated", "DH"), ("starting", "SP"), ("reliever", "RP"),
]

rows, pos = [], None
with open("../data/raw/mlbtr_2027_fa.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        m = re.match(r"^(.+?)\s+\((\d{2})\)\s*(?:[–—-]\s*(.*))?$", line)
        if m:
            if pos:
                rows.append({
                    "player_name": m.group(1).strip(),
                    "position": pos,
                    "age_2027": int(m.group(2)),
                    "option_note": (m.group(3) or "").strip(),
                })
        elif len(line) < 40:
            for key, val in pos_map:
                if key in line.lower():
                    pos = val

raw = pd.DataFrame(rows)

# A player listed under several positions becomes one row
fa = (raw.groupby("player_name", sort=False)
         .agg(position=("position", "first"),
              all_positions=("position", lambda s: "/".join(dict.fromkeys(s))),
              age_2027=("age_2027", "first"),
              option_note=("option_note", lambda s: next((x for x in s if x), "")))
         .reset_index())

fa["player_type"] = fa["position"].apply(lambda p: "pitcher" if p in ["SP", "RP"] else "hitter")
fa["has_option"] = fa["option_note"] != ""

print(len(raw), "lines ->", len(fa), "unique players")
print(fa.groupby("position").size())
print(fa["has_option"].sum(), "players with an option or opt-out noted")
fa.to_csv("../data/processed/fa_2027_list.csv", index=False)
fa.head(10)

272 lines -> 257 unique players
position
1B    10
2B    17
3B    10
C     15
CF    11
DH     6
LF    13
RF    13
RP    90
SP    64
SS     8
dtype: int64
64 players with an option or opt-out noted


,player_name,position,all_positions,age_2027,option_note,player_type,has_option
0,Travis d’Arnaud,C,C,38,,hitter,False
1,Elias Diaz,C,C,36,,hitter,False
2,Mitch Garver,C,C,36,,hitter,False
3,Austin Hedges,C,C,34,,hitter,False
4,Jonah Heim,C,C,32,,hitter,False
5,Kyle Higashioka,C,C,37,$7MM mutual option with a $1MM buyout,hitter,True
6,Ryan Jeffers,C,C,30,,hitter,False
7,Carson Kelly,C,C,32,$7.5MM mutual option with a $1.5MM buyout,hitter,True
8,Luke Maile,C,C,36,,hitter,False
9,James McCann,C,C,37,,hitter,False


In [27]:
import re, time, unicodedata
import numpy as np
import pandas as pd
import requests

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = s.replace("-", " ")
    s = re.sub(r"\b(jr|sr|ii|iii|iv)\b\.?", "", s)
    s = re.sub(r"[^a-z ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return {"michael fiers": "mike fiers", "steven pearce": "steve pearce"}.get(s, s)

fa = pd.read_csv("../data/processed/fa_2027_list.csv")
bh = pd.read_csv("../data/processed/bwar_hitters.csv")
bp = pd.read_csv("../data/processed/bwar_pitchers.csv")
sh = pd.read_csv("../data/processed/savant_hitters.csv")
sp = pd.read_csv("../data/processed/savant_pitchers.csv")

# Only the 2024-2026 seasons matter for the 2027 class
bh = bh[bh["year_ID"] >= 2024].copy()
bp = bp[bp["year_ID"] >= 2024].copy()
bh["key"] = bh["name"].apply(norm)
bp["key"] = bp["name"].apply(norm)
fa["key"] = fa["player_name"].apply(norm)

# 1. Candidate MLB IDs for each name
cands = {}
for t, df in [("hitter", bh), ("pitcher", bp)]:
    g = df.groupby("key")["mlb_ID"].unique()
    cands[t] = {k: [int(i) for i in v] for k, v in g.items()}
fa["cand_ids"] = [cands[t].get(k, []) for t, k in zip(fa["player_type"], fa["key"])]

# 2. Birth dates from the MLB Stats API
all_ids = sorted({i for c in fa["cand_ids"] for i in c})
births = {}
for i in range(0, len(all_ids), 100):
    batch = all_ids[i:i + 100]
    r = requests.get("https://statsapi.mlb.com/api/v1/people",
                     params={"personIds": ",".join(map(str, batch))}, timeout=30)
    r.raise_for_status()
    for p in r.json().get("people", []):
        if "birthDate" in p:
            births[p["id"]] = p["birthDate"]
    time.sleep(0.5)

# 3. If a name has several candidates, use the age from the MLBTR list to choose
def pick(row):
    c = row["cand_ids"]
    if len(c) == 1:
        return c[0]
    ok = []
    for i in c:
        if i in births:
            by = int(births[i][:4])
            if ((2027 - by) - row["age_2027"]) in (0, 1):
                ok.append(i)
    return ok[0] if len(ok) == 1 else np.nan

fa["mlb_ID"] = fa.apply(pick, axis=1)
fa["birth_date"] = pd.to_datetime(fa["mlb_ID"].map(births))
fa["age_at_signing"] = ((pd.Timestamp("2026-12-01") - fa["birth_date"]).dt.days / 365.25).round(1)

# 4. Weighted WAR over 2026, 2025, 2024 (same 5-4-3 weights as the training data)
war = {}
for t, df in [("hitter", bh), ("pitcher", bp)]:
    for mid, yr, w in zip(df["mlb_ID"], df["year_ID"], df["WAR"]):
        war[(t, int(mid), int(yr))] = w

for k, col in enumerate(["war_1", "war_2", "war_3"]):
    fa[col] = [war.get((t, int(i), 2026 - k), 0.0) if pd.notna(i) else np.nan
               for t, i in zip(fa["player_type"], fa["mlb_ID"])]
fa["war_w"] = (5 * fa["war_1"] + 4 * fa["war_2"] + 3 * fa["war_3"]) / 12

# 5. Savant percentiles, weighted the same way
feats = {
    "hitter": ["xwoba", "brl_percent", "exit_velocity", "chase_percent",
               "whiff_percent", "bb_percent", "sprint_speed"],
    "pitcher": ["xera", "fb_velocity", "k_percent", "bb_percent",
                "whiff_percent", "chase_percent", "hard_hit_percent"],
}
sav = {"hitter": sh, "pitcher": sp}

def savant_2027(df, t):
    s, f = sav[t], feats[t]
    lookup = dict(zip(zip(s["player_id"].astype(int), s["year"].astype(int)),
                      s[f].values.astype(float)))
    out = []
    for mid in df["mlb_ID"]:
        if pd.isna(mid):
            out.append([np.nan] * len(f))
            continue
        num, den = np.zeros(len(f)), np.zeros(len(f))
        for k, w in enumerate([5, 4, 3]):
            arr = lookup.get((int(mid), 2026 - k))
            if arr is None:
                continue
            ok = ~np.isnan(arr)
            num[ok] += w * arr[ok]
            den[ok] += w
        out.append(np.where(den > 0, num / np.where(den > 0, den, 1), np.nan))
    return pd.DataFrame(out, columns=f, index=df.index)

sv = pd.concat([savant_2027(fa[fa["player_type"] == t], t) for t in ["hitter", "pitcher"]])
fa = fa.join(sv)

# 6. Model inputs that match the training columns
fa["age_sq"] = (fa["age_at_signing"] - 31) ** 2
fa["year"] = 2027 - 2015
fa["qualifying_offer"] = 0   # placeholder until offers are made after the World Series

fa.drop(columns=["cand_ids"]).to_csv("../data/processed/fa_2027_features.csv", index=False)

# Summary
has_sav = np.where(fa["player_type"] == "hitter", fa["xwoba"].notna(), fa["xera"].notna())
print("Players:", len(fa))
print("Matched to an MLB ID:", fa["mlb_ID"].notna().sum())
print("With Savant data:", int(has_sav.sum()))
print("\nNo ID (no 2024-2026 MLB season, or an ambiguous name):")
print(fa[fa["mlb_ID"].isna()][["player_name", "position", "age_2027"]].to_string(index=False))
print("\nTop 10 by weighted WAR:")
fa.nlargest(10, "war_w")[["player_name", "position", "age_at_signing", "war_3", "war_2", "war_1", "war_w"]]

Players: 257
Matched to an MLB ID: 253
With Savant data: 212

No ID (no 2024-2026 MLB season, or an ambiguous name):
   player_name position  age_2027
Kiké Hernandez       3B        35
   Kyle Wright       SP        31
Trevor Stephan       RP        31
    Rowan Wick       RP        34

Top 10 by weighted WAR:


,player_name,position,age_at_signing,war_3,war_2,war_1,war_w
157,Tarik Skubal,SP,30.0,6.66,6.39,4.60,5.711667
61,Randy Arozarena,LF,31.8,1.71,3.87,5.70,4.092500
95,Seiya Suzuki,RF,32.3,3.53,2.58,5.12,3.875833
149,Drew Rasmussen,SP,31.3,0.60,4.40,4.66,3.558333
66,Ian Happ,LF,32.3,3.92,3.96,2.86,3.491667
33,Brandon Lowe,2B,32.4,2.35,1.81,5.35,3.420000
135,Nick Martinez,SP,36.3,3.98,2.19,3.91,3.354167
130,Michael King,SP,31.5,4.07,1.19,4.26,3.189167
152,Trevor Rogers,SP,29.0,0.59,5.28,2.23,2.836667
28,Jazz Chisholm Jr.,2B,28.8,2.92,4.12,1.69,2.807500


In [33]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

feats = {
    "hitter": ["xwoba", "brl_percent", "exit_velocity", "chase_percent",
               "whiff_percent", "bb_percent", "sprint_speed"],
    "pitcher": ["xera", "fb_velocity", "k_percent", "bb_percent",
                "whiff_percent", "chase_percent", "hard_hit_percent"],
}
sav = {
    "hitter": pd.read_csv("../data/processed/savant_hitters.csv"),
    "pitcher": pd.read_csv("../data/processed/savant_pitchers.csv"),
}

# Training contracts ($5M+, Ohtani excluded, qualifying-offer acceptances removed)
d = pd.read_csv("../data/processed/contracts_model.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d = d[d["total_value"] >= 5_000_000].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2
d["year"] = d["signing_year"] - 2015

# 2027 free agents
fa = pd.read_csv("../data/processed/fa_2027_features.csv")

def savant_avg(df, t, year_col):
    s, f = sav[t], feats[t]
    lookup = dict(zip(zip(s["player_id"].astype(int), s["year"].astype(int)),
                      s[f].values.astype(float)))
    out = []
    for mid, ly in zip(df["mlb_ID"], df[year_col]):
        if pd.isna(mid):
            out.append([np.nan] * len(f))
            continue
        num, den = np.zeros(len(f)), np.zeros(len(f))
        for k, w in enumerate([5, 4, 3]):
            arr = lookup.get((int(mid), int(ly) - k))
            if arr is None:
                continue
            ok = ~np.isnan(arr)
            num[ok] += w * arr[ok]
            den[ok] += w
        out.append(np.where(den > 0, num / np.where(den > 0, den, 1), np.nan))
    return pd.DataFrame(out, columns=f, index=df.index)

def build(df, cols):
    X = pd.get_dummies(df[cols + ["position"]], columns=["position"], dtype=float)
    resid = []
    for yr in range(2021, 2027):          # rolling out-of-sample errors
        tr = (df["signing_year"] < yr).values
        te = (df["signing_year"] == yr).values
        if te.sum() == 0 or tr.sum() < 30:
            continue
        m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
        m.fit(X[tr], df.loc[tr, "aav_m"])
        resid.extend((df.loc[te, "aav_m"] - m.predict(X[te])).tolist())
    final = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    final.fit(X, df["aav_m"])             # final model uses ALL years
    lo, hi = np.percentile(resid, [10, 90])
    return final, list(X.columns), lo, hi

def predict(model, xcols, df, cols):
    X = pd.get_dummies(df[cols + ["position"]], columns=["position"], dtype=float)
    return model.predict(X.reindex(columns=xcols, fill_value=0.0))

base = ["age_at_signing", "age_sq", "qualifying_offer", "year"]
results, bands = [], {}

for t in ["hitter", "pitcher"]:
    f = feats[t]
    tr = d[d["player_type"] == t].copy()
    tr = pd.concat([tr, savant_avg(tr, t, "stat_year")], axis=1)

    colsA = base + ["war_w"]                       # Model A: WAR only
    mA, xA, loA, hiA = build(tr, colsA)

    trB = tr[tr[f[0]].notna()].copy()              # Model B: WAR + Savant
    trB[f] = trB[f].fillna(50)
    colsB = base + ["war_w"] + f
    mB, xB, loB, hiB = build(trB, colsB)
    bands[f"{t} WAR only"] = (loA, hiA)
    bands[f"{t} WAR + Savant"] = (loB, hiB)

    te = fa[(fa["player_type"] == t) & fa["mlb_ID"].notna()
            & fa["age_at_signing"].notna()].copy()
    has_sav = te[f[0]].notna().values
    te[f] = te[f].fillna(50)

    pA = predict(mA, xA, te, colsA)
    pB = predict(mB, xB, te, colsB)
    te["pred_aav_m"] = np.where(has_sav, pB, pA)
    te["low_m"] = (np.where(has_sav, loB, loA) + te["pred_aav_m"]).clip(lower=0)
    te["high_m"] = np.where(has_sav, hiB, hiA) + te["pred_aav_m"]
    te["model"] = np.where(has_sav, "WAR + Savant", "WAR only")
    results.append(te)

out = pd.concat(results)
out["below_5m_floor"] = out["pred_aav_m"] < 5
out["note"] = np.where(out["player_name"] == "Tarik Skubal",
                       "outlier: model likely far too low", "")

show = ["player_name", "position", "age_at_signing", "war_w", "pred_aav_m",
        "low_m", "high_m", "model", "option_note", "below_5m_floor", "note"]
out = out.sort_values("pred_aav_m", ascending=False)[show].round(1)
out.to_csv("../outputs/fa_2027_predictions.csv", index=False)

print("Predicted:", len(out), "of", len(fa), "players")
print(out["model"].value_counts().to_string())
print("\nRange used (past errors in $M, 10th to 90th percentile):")
for k, (lo, hi) in bands.items():
    print(f"  {k}: {lo:+.1f} to {hi:+.1f}")
print("\nUnder the $5M floor:", int(out["below_5m_floor"].sum()), "players")
out.head(15)

Predicted: 253 of 257 players
model
WAR + Savant    212
WAR only         41

Range used (past errors in $M, 10th to 90th percentile):
  hitter WAR only: -4.3 to +7.8
  hitter WAR + Savant: -4.1 to +6.7
  pitcher WAR only: -4.4 to +6.1
  pitcher WAR + Savant: -4.2 to +6.1

Under the $5M floor: 49 players


,player_name,position,age_at_signing,war_w,pred_aav_m,low_m,high_m,model,option_note,below_5m_floor,note
157,Tarik Skubal,SP,30.0,5.7,32.2,28.1,38.3,WAR + Savant,NaN,False,outlier: model likely far too low
95,Seiya Suzuki,RF,32.3,3.9,24.0,19.9,30.7,WAR + Savant,NaN,False,
149,Drew Rasmussen,SP,31.3,3.6,22.9,18.7,29.0,WAR + Savant,$20MM club option with a $500K buyout,False,
61,Randy Arozarena,LF,31.8,4.1,21.3,17.2,28.0,WAR + Savant,NaN,False,
84,Ronald Acuña Jr.,RF,29.0,1.8,20.8,16.7,27.5,WAR + Savant,$17MM club option with a $10MM buyout,False,
143,Freddy Peralta,SP,30.5,2.8,19.3,15.2,25.4,WAR + Savant,NaN,False,
66,Ian Happ,LF,32.3,3.5,19.2,15.1,25.9,WAR + Savant,NaN,False,
130,Michael King,SP,31.5,3.2,19.1,14.9,25.1,WAR + Savant,"can opt out of remaining two years, $58MM for ...",False,
74,Trent Grisham,CF,30.1,1.5,19.0,14.9,25.7,WAR + Savant,NaN,False,
152,Trevor Rogers,SP,29.0,2.8,18.2,14.0,24.2,WAR + Savant,NaN,False,


In [34]:
import pandas as pd

c = pd.read_csv("../data/processed/contracts_with_age.csv")
sub = c[(c["player_type"] == "hitter") & (c["signing_year"] >= 2025)
        & c["war_w"].between(1.3, 2.3) & (c["total_value"] >= 5_000_000)].copy()
sub["aav_m"] = (sub["aav"] / 1e6).round(1)

print("Comparable hitters:", len(sub))
print("Median AAV: $%.1fM" % sub["aav_m"].median())
print("Middle half: $%.1fM to $%.1fM" % (sub["aav_m"].quantile(.25), sub["aav_m"].quantile(.75)))
sub.sort_values("aav_m", ascending=False)[
    ["player_name", "signing_year", "age_at_signing", "war_w", "aav_m"]].head(15)

Comparable hitters: 18
Median AAV: $10.1M
Middle half: $7.0M to $14.4M


,player_name,signing_year,age_at_signing,war_w,aav_m
639,Trent Grisham,2026,29.1,2.034167,22.0
635,Jorge Polanco,2026,32.4,1.940833,20.0
566,Joc Pederson,2025,32.6,1.712500,18.5
564,Tyler O'Neill,2025,29.4,1.544167,16.5
637,Ryan O'Hearn,2026,32.4,1.924167,14.5
565,Jurickson Profar,2025,31.8,1.820000,14.0
649,Luis Arraez,2026,28.6,2.069167,12.0
573,Carlos Santana,2025,38.7,2.198333,12.0
641,Harrison Bader,2026,31.5,2.085000,10.2
576,Max Kepler,2025,31.8,1.771667,10.0


In [35]:
import pandas as pd

c = pd.read_csv("../data/processed/contracts_with_age.csv")
qo1 = c[(c["qualifying_offer"] == 1) & (c["years"] == 1)].copy()
qo1["aav_m"] = (qo1["aav"] / 1e6).round(2)

print("One-year deals with a qualifying offer flag, by class:")
print(qo1.groupby("signing_year")["aav_m"].agg(["count", "min", "max"]))
print()
print(qo1[qo1["signing_year"] >= 2024]
      .sort_values(["signing_year", "aav_m"], ascending=[True, False])
      [["player_name", "signing_year", "war_w", "aav_m"]].to_string(index=False))

One-year deals with a qualifying offer flag, by class:
              count    min    max
signing_year                     
2016              6   8.00  15.80
2017              3  17.20  18.50
2018              3   6.50  14.00
2019              3  17.90  21.02
2020              2  17.80  18.00
2021              2  18.90  18.90
2022              2  18.40  21.00
2023              2  19.65  19.65
2025              1  21.05  21.05
2026              5  22.02  22.02

     player_name  signing_year    war_w  aav_m
   Nick Martinez          2025 2.515833  21.05
   Trent Grisham          2026 2.034167  22.02
  Gleyber Torres          2026 2.615000  22.02
Brandon Woodruff          2026 1.101667  22.02
   Shota Imanaga          2026 1.620000  22.02
      Zac Gallen          2026 2.360833  22.02


In [36]:
import pandas as pd

c = pd.read_csv("../data/processed/contracts_with_age.csv")

# League-set qualifying offer amounts ($M) by signing class.
# 2025 and 2026 are confirmed by your own output; the earlier years are from my memory,
# so check the printout below: each year's dropped players should share one salary.
qo_amount = {2016: 15.8, 2017: 17.2, 2018: 17.4, 2019: 17.9, 2020: 17.8,
             2021: 18.9, 2022: 18.4, 2023: 19.65, 2024: 20.3, 2025: 21.05, 2026: 22.02}

aav_m = c["aav"] / 1e6
amt = c["signing_year"].map(qo_amount)
accepted = (c["qualifying_offer"] == 1) & (c["years"] == 1) & ((aav_m - amt).abs() < 0.06)

print("Dropping", accepted.sum(), "qualifying-offer acceptances:")
print(c[accepted].assign(aav_m=aav_m[accepted].round(2))
      .sort_values("signing_year")[["player_name", "signing_year", "aav_m"]].to_string(index=False))

c[~accepted].to_csv("../data/processed/contracts_model.csv", index=False)
print("Saved contracts_model.csv with", (~accepted).sum(), "contracts")

Dropping 18 qualifying-offer acceptances:
      player_name  signing_year  aav_m
     Matt Wieters          2016  15.80
     Colby Rasmus          2016  15.80
   Brett Anderson          2016  15.80
      Neil Walker          2017  17.20
Jeremy Hellickson          2017  17.20
     Hyun-Jin Ryu          2019  17.90
       Jose Abreu          2020  17.80
    Kevin Gausman          2021  18.90
   Marcus Stroman          2021  18.90
     Brandon Belt          2022  18.40
     Joc Pederson          2023  19.65
     Martín Pérez          2023  19.65
    Nick Martinez          2025  21.05
    Trent Grisham          2026  22.02
   Gleyber Torres          2026  22.02
 Brandon Woodruff          2026  22.02
    Shota Imanaga          2026  22.02
       Zac Gallen          2026  22.02
Saved contracts_model.csv with 1600 contracts


In [37]:
old = pd.read_csv("../outputs/fa_2027_predictions_v1.csv")
new = pd.read_csv("../outputs/fa_2027_predictions.csv")
cmp = old[["player_name", "pred_aav_m"]].merge(
    new[["player_name", "pred_aav_m"]], on="player_name", suffixes=("_old", "_new"))
cmp["change"] = (cmp["pred_aav_m_new"] - cmp["pred_aav_m_old"]).round(1)
print("Average change: %+.1fM" % cmp["change"].mean())
print(cmp.sort_values("pred_aav_m_old", ascending=False).head(15).to_string(index=False))

Average change: -0.1M
     player_name  pred_aav_m_old  pred_aav_m_new  change
    Tarik Skubal            31.8            32.2     0.4
    Seiya Suzuki            23.8            24.0     0.2
  Drew Rasmussen            22.6            22.9     0.3
 Randy Arozarena            21.3            21.3     0.0
Ronald Acuña Jr.            20.8            20.8     0.0
        Ian Happ            19.2            19.2     0.0
  Freddy Peralta            19.2            19.3     0.1
   Trent Grisham            18.9            19.0     0.1
    Michael King            18.9            19.1     0.2
   Trevor Rogers            18.2            18.2     0.0
    Ryan Jeffers            18.0            18.1     0.1
    Nick Pivetta            17.8            18.0     0.2
   J.P. Crawford            17.6            17.7     0.1
   Nick Martinez            17.3            17.4     0.1
      Sonny Gray            16.8            16.9     0.1


In [38]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = pd.read_csv("../data/processed/contracts_model.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")]
d = d[(d["total_value"] >= 5_000_000) & (d["player_type"] == "hitter")].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2
d["year"] = d["signing_year"] - 2015

cols = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "year", "position"]
X = pd.get_dummies(d[cols], columns=["position"], dtype=float)

rows = []
for yr in range(2021, 2027):
    tr = (d["signing_year"] < yr).values
    te = (d["signing_year"] == yr).values
    m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    m.fit(X[tr], d.loc[tr, "aav_m"])
    t = d[te].copy()
    t["pred"] = m.predict(X[te])
    rows.append(t)

r = pd.concat(rows)
r["err"] = r["pred"] - r["aav_m"]          # positive = model guessed too high
r["war_band"] = pd.cut(r["war_w"], [-10, 0.5, 1.3, 2.3, 3.5, 20],
                       labels=["under 0.5", "0.5-1.3", "1.3-2.3", "2.3-3.5", "3.5+"])

for label, sub in [("All test years (2021-2026)", r),
                   ("2025-2026 only", r[r["signing_year"] >= 2025])]:
    print(label)
    print(sub.groupby("war_band", observed=True).agg(
        n=("err", "size"), actual=("aav_m", "mean"),
        predicted=("pred", "mean"), bias=("err", "mean")).round(1))
    print()

All test years (2021-2026)
            n  actual  predicted  bias
war_band                              
under 0.5   8     6.8        5.7  -1.2
0.5-1.3    53     7.2        8.2   1.0
1.3-2.3    56    11.0       10.9  -0.1
2.3-3.5    38    17.5       15.3  -2.1
3.5+       23    28.1       21.7  -6.4

2025-2026 only
            n  actual  predicted  bias
war_band                              
under 0.5   2     6.4        7.6   1.2
0.5-1.3    15     7.0        9.1   2.0
1.3-2.3    17    10.6       12.0   1.4
2.3-3.5    16    19.9       17.8  -2.2
3.5+        8    33.2       23.3  -9.9



In [39]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d0 = pd.read_csv("../data/processed/contracts_model.csv")
d0 = d0[d0["age_at_signing"].notna() & (d0["player_name"] != "Shohei Ohtani")]
d0 = d0[d0["total_value"] >= 5_000_000].copy()
d0["aav_m"] = d0["aav"] / 1e6
d0["age_sq"] = (d0["age_at_signing"] - 31) ** 2
d0["year"] = d0["signing_year"] - 2015
d0["war_sq"] = d0["war_w"].clip(lower=0) ** 2

def rolling(df, cols):
    X = pd.get_dummies(df[cols + ["position"]], columns=["position"], dtype=float)
    rows = []
    for yr in range(2021, 2027):
        tr = (df["signing_year"] < yr).values
        te = (df["signing_year"] == yr).values
        m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
        m.fit(X[tr], df.loc[tr, "aav_m"])
        t = df[te].copy()
        t["pred"] = m.predict(X[te])
        rows.append(t)
    r = pd.concat(rows)
    r["err"] = r["pred"] - r["aav_m"]
    return r

base = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "year"]
for t in ["hitter", "pitcher"]:
    df = d0[d0["player_type"] == t]
    print(t.upper() + "S")
    for name, cols in [("Linear WAR", base), ("+ WAR squared", base + ["war_sq"])]:
        r = rolling(df, cols)
        hi = r[r["war_w"] >= 3.5]
        mid = r[(r["war_w"] >= 2.3) & (r["war_w"] < 3.5)]
        print(f"  {name:14s} MAE ${r['err'].abs().mean():.2f}M"
              f" | bias 3.5+ WAR: ${hi['err'].mean():+.1f}M (n={len(hi)})"
              f" | bias 2.3-3.5: ${mid['err'].mean():+.1f}M (n={len(mid)})")

HITTERS
  Linear WAR     MAE $3.83M | bias 3.5+ WAR: $-6.4M (n=23) | bias 2.3-3.5: $-2.1M (n=38)
  + WAR squared  MAE $3.88M | bias 3.5+ WAR: $-6.0M (n=23) | bias 2.3-3.5: $-2.8M (n=38)
PITCHERS
  Linear WAR     MAE $3.47M | bias 3.5+ WAR: $-5.2M (n=10) | bias 2.3-3.5: $-1.9M (n=25)
  + WAR squared  MAE $3.44M | bias 3.5+ WAR: $-5.0M (n=10) | bias 2.3-3.5: $-2.6M (n=25)


In [40]:
import pandas as pd

out = pd.read_csv("../outputs/fa_2027_predictions.csv")
notes = {
    "Tarik Skubal": "outlier: model likely far too low",
    "Ronald Acuña Jr.": "former MVP; 2024-2026 window includes injury-shortened seasons; "
                        "model likely too low if he reached the market, but a club option makes that unlikely",
}
out["note"] = out["player_name"].map(notes).fillna(out["note"])
out.to_csv("../outputs/fa_2027_predictions.csv", index=False)
out[out["note"].notna()][["player_name", "pred_aav_m", "note"]]

,player_name,pred_aav_m,note
0,Tarik Skubal,32.2,outlier: model likely far too low
4,Ronald Acuña Jr.,20.8,former MVP; 2024-2026 window includes injury-s...


In [41]:
fa = pd.read_csv("../data/processed/fa_2027_features.csv")
fa[fa["player_name"].str.contains("Acuña")][["player_name", "war_3", "war_2", "war_1", "war_w"]]

,player_name,war_3,war_2,war_1,war_w
84,Ronald Acuña Jr.,0.02,3.05,1.89,1.809167


In [42]:
bh = pd.read_csv("../data/processed/bwar_hitters.csv")
bh[bh["name"].str.contains("Ronald Acu")][["name", "year_ID", "PA", "WAR"]].tail(6)

,name,year_ID,PA,WAR
87719,Ronald Acuña Jr.,2021,360,3.47
87720,Ronald Acuña Jr.,2022,533,2.33
87721,Ronald Acuña Jr.,2023,735,8.40
87722,Ronald Acuña Jr.,2024,222,0.02
87723,Ronald Acuña Jr.,2025,412,3.05
87724,Ronald Acuña Jr.,2026,469,1.89


In [1]:
import pandas as pd

out = pd.read_csv("../outputs/fa_2027_predictions.csv")
notes = {
    "Tarik Skubal": "outlier: model likely far too low",
    "Ronald Acuña Jr.": "former MVP; 222/412/469 PA in 2024-2026, and his 2023 season (8.4 WAR) is outside the model's window; "
                        "a $17MM club option makes reaching the market unlikely",
}
out["note"] = out["player_name"].map(notes).fillna(out["note"])
out.to_csv("../outputs/fa_2027_predictions.csv", index=False)
out[out["note"].notna()][["player_name", "pred_aav_m", "note"]]

,player_name,pred_aav_m,note
0,Tarik Skubal,32.2,outlier: model likely far too low
4,Ronald Acuña Jr.,20.8,"former MVP; 222/412/469 PA in 2024-2026, and h..."


In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d0 = pd.read_csv("../data/processed/contracts_model.csv")
d0 = d0[d0["age_at_signing"].notna() & (d0["player_name"] != "Shohei Ohtani")]
d0 = d0[d0["total_value"] >= 5_000_000].copy()
d0["aav_m"] = d0["aav"] / 1e6
d0["age_sq"] = (d0["age_at_signing"] - 31) ** 2
d0["year"] = d0["signing_year"] - 2015
d0["war_sq"] = d0["war_w"].clip(lower=0) ** 2

def rolling(df, cols):
    X = pd.get_dummies(df[cols + ["position"]], columns=["position"], dtype=float)
    rows = []
    for yr in range(2021, 2027):
        tr = (df["signing_year"] < yr).values
        te = (df["signing_year"] == yr).values
        m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
        m.fit(X[tr], df.loc[tr, "aav_m"])
        t = df[te].copy()
        t["pred"] = m.predict(X[te])
        rows.append(t)
    r = pd.concat(rows)
    r["err"] = r["pred"] - r["aav_m"]
    return r

base = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "year"]
for t in ["hitter", "pitcher"]:
    df = d0[d0["player_type"] == t]
    print(t.upper() + "S")
    for name, cols in [("Linear WAR", base), ("+ WAR squared", base + ["war_sq"])]:
        r = rolling(df, cols)
        hi = r[r["war_w"] >= 3.5]
        mid = r[(r["war_w"] >= 2.3) & (r["war_w"] < 3.5)]
        print(f"  {name:14s} MAE ${r['err'].abs().mean():.2f}M"
              f" | bias 3.5+ WAR: ${hi['err'].mean():+.1f}M (n={len(hi)})"
              f" | bias 2.3-3.5: ${mid['err'].mean():+.1f}M (n={len(mid)})")

HITTERS
  Linear WAR     MAE $3.83M | bias 3.5+ WAR: $-6.4M (n=23) | bias 2.3-3.5: $-2.1M (n=38)
  + WAR squared  MAE $3.88M | bias 3.5+ WAR: $-6.0M (n=23) | bias 2.3-3.5: $-2.8M (n=38)
PITCHERS
  Linear WAR     MAE $3.47M | bias 3.5+ WAR: $-5.2M (n=10) | bias 2.3-3.5: $-1.9M (n=25)
  + WAR squared  MAE $3.44M | bias 3.5+ WAR: $-5.0M (n=10) | bias 2.3-3.5: $-2.6M (n=25)


In [3]:
import pandas as pd

c = pd.read_csv("../data/processed/contracts_with_age.csv")

# League-set qualifying offer amounts ($M) by signing class.
# 2025 and 2026 are confirmed by your own output; the earlier years are from memory.
qo_amount = {2016: 15.8, 2017: 17.2, 2018: 17.4, 2019: 17.9, 2020: 17.8,
             2021: 18.9, 2022: 18.4, 2023: 19.65, 2024: 20.3, 2025: 21.05, 2026: 22.02}

aav_m = c["aav"] / 1e6
amt = c["signing_year"].map(qo_amount)
one_year_qo = (c["qualifying_offer"] == 1) & (c["years"] == 1)
accepted = one_year_qo & ((aav_m - amt).abs() < 0.06)

print("Dropping", accepted.sum(), "qualifying-offer acceptances:")
print(c[accepted].assign(aav_m=aav_m[accepted].round(2))
      .sort_values("signing_year")[["player_name", "signing_year", "aav_m"]].to_string(index=False))

kept = one_year_qo & ~accepted
print("\nOne-year deals with a QO flag that were KEPT (check these):")
print(c[kept].assign(aav_m=aav_m[kept].round(2))
      .sort_values("signing_year")[["player_name", "signing_year", "aav_m"]].to_string(index=False))

c[~accepted].to_csv("../data/processed/contracts_model.csv", index=False)
print("\nSaved contracts_model.csv with", (~accepted).sum(), "contracts")

Dropping 18 qualifying-offer acceptances:
      player_name  signing_year  aav_m
     Matt Wieters          2016  15.80
     Colby Rasmus          2016  15.80
   Brett Anderson          2016  15.80
      Neil Walker          2017  17.20
Jeremy Hellickson          2017  17.20
     Hyun-Jin Ryu          2019  17.90
       Jose Abreu          2020  17.80
    Kevin Gausman          2021  18.90
   Marcus Stroman          2021  18.90
     Brandon Belt          2022  18.40
     Joc Pederson          2023  19.65
     Martín Pérez          2023  19.65
    Nick Martinez          2025  21.05
    Trent Grisham          2026  22.02
   Gleyber Torres          2026  22.02
 Brandon Woodruff          2026  22.02
    Shota Imanaga          2026  22.02
       Zac Gallen          2026  22.02

One-year deals with a QO flag that were KEPT (check these):
     player_name  signing_year  aav_m
   Dexter Fowler          2016  13.00
     Ian Desmond          2016   8.00
 Hisashi Iwakuma          2016  12.00
   J

In [5]:
import os
import re
import pandas as pd

num_words = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
             "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12,
             "thirteen": 13, "fourteen": 14, "fifteen": 15}

pat = re.compile(
    r"^\s*(\d{1,2})\.\s+(.+?),\s*([A-Z0-9/]+):\s*(\d+|[A-Za-z]+)[\s-]+years?,\s*\$([\d.]+)\s*MM",
    re.IGNORECASE)

cols = ["player_name", "signing_year", "mlbtr_rank", "mlbtr_pos", "mlbtr_years", "mlbtr_total_m"]

def parse_mlbtr(path, signing_year):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            m = pat.match(line)
            if not m:
                continue
            rank, name, pos, yrs, total = m.groups()
            yrs = int(yrs) if yrs.isdigit() else num_words.get(yrs.lower())
            if yrs is None:
                print("Could not read years in:", line.strip()[:80])
                continue
            rows.append([name.strip(), signing_year, int(rank), pos.upper(), yrs, float(total)])
    out = pd.DataFrame(rows, columns=cols)
    out["mlbtr_aav_m"] = (out["mlbtr_total_m"] / out["mlbtr_years"]).round(2)
    return out

frames = []
for yr in [2024, 2025, 2026]:
    path = f"../data/raw/mlbtr_pred_{yr}.txt"
    if not os.path.exists(path):
        print(yr, "- file not found yet, skipping")
        continue
    df_yr = parse_mlbtr(path, yr)
    print(yr, "-", len(df_yr), "predictions parsed")
    frames.append(df_yr)

if frames:
    mlbtr = pd.concat(frames, ignore_index=True)
    mlbtr.to_csv("../data/processed/mlbtr_predictions.csv", index=False)
    display(mlbtr[mlbtr["signing_year"] == 2026].head(6))

2024 - 0 predictions parsed
2025 - 0 predictions parsed
2026 - 50 predictions parsed


,player_name,signing_year,mlbtr_rank,mlbtr_pos,mlbtr_years,mlbtr_total_m,mlbtr_aav_m
0,Kyle Tucker,2026,1,RF,11,400.0,36.36
1,Bo Bichette,2026,2,SS,8,208.0,26.0
2,Dylan Cease,2026,3,SP,7,189.0,27.0
3,Munetaka Murakami,2026,4,3B/1B,8,180.0,22.5
4,Alex Bregman,2026,5,3B,6,160.0,26.67
5,Framber Valdez,2026,6,SP,5,150.0,30.0


In [6]:
for yr in [2024, 2025]:
    with open(f"../data/raw/mlbtr_pred_{yr}.txt", encoding="utf-8") as f:
        text = f.read()
    lines = [l for l in text.splitlines() if l.strip()]
    print(f"{yr}: {len(text)} characters, {len(lines)} non-empty lines")
    for l in lines[:6]:
        print("   ", repr(l[:100]))
    print()

2024: 125816 characters, 426 non-empty lines
    '1. Shohei Ohtani. Twelve years, $528MM'
    'Tim Dierkes’ prediction: Dodgers / Anthony Franco’s prediction: Dodgers / Darragh McDonald’s predict'
    'Ohtani’s two-way legend stretches back to his days in Japan. He’d hit .286/.358/.501 while pitching '
    'Ohtani was arguably the most anticipated international signee ever. There was never any question the'
    'He dispelled some concern with a strong rookie showing. Ohtani connected on 22 homers in 114 games t'
    'Ohtani had a fine second year as a hitter. His return to the mound in the shortened 2020 campaign wa'

2025: 128137 characters, 372 non-empty lines
    '1. Juan Soto: Thirteen years, $600MM'
    'Tim Dierkes’ prediction: Yankees / Anthony Franco’s prediction: Yankees / Darragh McDonald’s predict'
    'Soto isn’t just one of the best hitters on the planet, he’s one of the youngest and thus most appeal'
    'It’s rare for a player to even make his big league debut at 19, let

In [7]:
import os
import re
import pandas as pd

num_words = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
             "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12,
             "thirteen": 13, "fourteen": 14, "fifteen": 15}

# rank. Name[, POS] then ":" or "." then "N years, $XMM" (years may be a word)
pat = re.compile(
    r"^\s*(\d{1,2})\.\s+(.+?)(?:,\s*([A-Z0-9/]+))?\s*[.:]\s*(\d+|[A-Za-z]+)[\s-]+years?,\s*\$([\d.,]+)\s*MM",
    re.IGNORECASE)

cols = ["player_name", "signing_year", "mlbtr_rank", "mlbtr_pos", "mlbtr_years", "mlbtr_total_m"]

def parse_mlbtr(path, signing_year):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            m = pat.match(line)
            if not m:
                continue
            rank, name, pos, yrs, total = m.groups()
            yrs = int(yrs) if yrs.isdigit() else num_words.get(yrs.lower())
            if yrs is None:
                print("Could not read years in:", line.strip()[:80])
                continue
            rows.append([name.strip(), signing_year, int(rank), (pos or "").upper(),
                         yrs, float(total.replace(",", ""))])
    out = pd.DataFrame(rows, columns=cols)
    out["mlbtr_aav_m"] = (out["mlbtr_total_m"] / out["mlbtr_years"]).round(2)
    return out

frames = []
for yr in [2024, 2025, 2026]:
    path = f"../data/raw/mlbtr_pred_{yr}.txt"
    if not os.path.exists(path):
        print(yr, "- file not found yet, skipping")
        continue
    df_yr = parse_mlbtr(path, yr)
    print(yr, "-", len(df_yr), "predictions parsed", "(expected 50)" if len(df_yr) != 50 else "")
    frames.append(df_yr)

if frames:
    mlbtr = pd.concat(frames, ignore_index=True)
    mlbtr.to_csv("../data/processed/mlbtr_predictions.csv", index=False)
    display(mlbtr.groupby("signing_year").head(3))

2024 - 50 predictions parsed 
2025 - 50 predictions parsed 
2026 - 50 predictions parsed 


,player_name,signing_year,mlbtr_rank,mlbtr_pos,mlbtr_years,mlbtr_total_m,mlbtr_aav_m
0,Shohei Ohtani,2024,1,,12,528.0,44.00
1,Cody Bellinger,2024,2,,12,264.0,22.00
2,Yoshinobu Yamamoto,2024,3,,9,225.0,25.00
50,Juan Soto,2025,1,,13,600.0,46.15
51,Corbin Burnes,2025,2,,7,200.0,28.57
52,Alex Bregman,2025,3,,7,182.0,26.00
100,Kyle Tucker,2026,1,RF,11,400.0,36.36
101,Bo Bichette,2026,2,SS,8,208.0,26.00
102,Dylan Cease,2026,3,SP,7,189.0,27.00


In [8]:
import re, unicodedata
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = s.replace("-", " ")
    s = re.sub(r"\b(jr|sr|ii|iii|iv)\b\.?", "", s)
    s = re.sub(r"[^a-z ]", "", s)
    return re.sub(r"\s+", " ", s).strip()

feats = {
    "hitter": ["xwoba", "brl_percent", "exit_velocity", "chase_percent",
               "whiff_percent", "bb_percent", "sprint_speed"],
    "pitcher": ["xera", "fb_velocity", "k_percent", "bb_percent",
                "whiff_percent", "chase_percent", "hard_hit_percent"],
}
sav = {
    "hitter": pd.read_csv("../data/processed/savant_hitters.csv"),
    "pitcher": pd.read_csv("../data/processed/savant_pitchers.csv"),
}

d = pd.read_csv("../data/processed/contracts_model.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d = d[d["total_value"] >= 5_000_000].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2
d["year"] = d["signing_year"] - 2015
d["key"] = d["player_name"].apply(norm)

def savant_avg(df, t):
    s, f = sav[t], feats[t]
    lookup = dict(zip(zip(s["player_id"].astype(int), s["year"].astype(int)),
                      s[f].values.astype(float)))
    out = []
    for mid, sy in zip(df["mlb_ID"], df["stat_year"]):
        if pd.isna(mid):
            out.append([np.nan] * len(f))
            continue
        num, den = np.zeros(len(f)), np.zeros(len(f))
        for k, w in enumerate([5, 4, 3]):
            arr = lookup.get((int(mid), int(sy) - k))
            if arr is None:
                continue
            ok = ~np.isnan(arr)
            num[ok] += w * arr[ok]
            den[ok] += w
        out.append(np.where(den > 0, num / np.where(den > 0, den, 1), np.nan))
    return pd.DataFrame(out, columns=f, index=df.index)

def new_model():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))

def oos_predictions(t):
    f = feats[t]
    df = d[d["player_type"] == t].copy()
    df = pd.concat([df, savant_avg(df, t)], axis=1)
    has_sav = df[f[0]].notna().values
    df[f] = df[f].fillna(50)
    colsA = ["age_at_signing", "age_sq", "qualifying_offer", "year", "war_w"]
    colsB = colsA + f
    XA = pd.get_dummies(df[colsA + ["position"]], columns=["position"], dtype=float)
    XB = pd.get_dummies(df[colsB + ["position"]], columns=["position"], dtype=float)
    out = []
    for yr in [2024, 2025, 2026]:
        tr = (df["signing_year"] < yr).values
        te = (df["signing_year"] == yr).values
        mA = new_model().fit(XA[tr], df.loc[tr, "aav_m"])
        mB = new_model().fit(XB[tr & has_sav], df.loc[tr & has_sav, "aav_m"])
        part = df[te].copy()
        part["model_aav_m"] = np.where(has_sav[te], mB.predict(XB[te]), mA.predict(XA[te]))
        out.append(part)
    return pd.concat(out)

oos = pd.concat([oos_predictions("hitter"), oos_predictions("pitcher")])
oos = (oos.rename(columns={"aav_m": "actual_aav_m", "years": "actual_years",
                           "total_value": "actual_total"})
          [["key", "signing_year", "actual_aav_m", "actual_years", "actual_total", "model_aav_m"]]
          .drop_duplicates(["key", "signing_year"]))
oos["actual_total_m"] = oos["actual_total"] / 1e6

mt = pd.read_csv("../data/processed/mlbtr_predictions.csv")
mt["key"] = mt["player_name"].apply(norm)
cmp = mt.merge(oos, on=["key", "signing_year"], how="left")

print("MLBTR top 50 -> players with an actual contract the model can score:")
for yr, g in cmp.groupby("signing_year"):
    print(f"  {yr}: {g['actual_aav_m'].notna().sum()} of {len(g)}")
    print("     not scored:", ", ".join(g.loc[g["actual_aav_m"].isna(), "player_name"]))

m = cmp[cmp["actual_aav_m"].notna()].copy()
m["mlbtr_err"] = m["mlbtr_aav_m"] - m["actual_aav_m"]
m["model_err"] = m["model_aav_m"] - m["actual_aav_m"]

def summarize(g):
    return pd.Series({
        "n": len(g),
        "MLBTR MAE": g["mlbtr_err"].abs().mean(),
        "Model MAE": g["model_err"].abs().mean(),
        "MLBTR bias": g["mlbtr_err"].mean(),
        "Model bias": g["model_err"].mean(),
    })

print("\nAAV error in $M (bias = predicted minus actual, negative = too low):")
print(m.groupby("signing_year").apply(summarize).round(2))
print("\nAll three classes pooled:")
print(summarize(m).round(2).to_string())

print("\nMLBTR on contract structure (for context):")
print("  average miss on years: %.1f" % (m["mlbtr_years"] - m["actual_years"]).abs().mean())
print("  average miss on total value: $%.0fM" % (m["mlbtr_total_m"] - m["actual_total_m"]).abs().mean())

m.to_csv("../outputs/benchmark_mlbtr_vs_model.csv", index=False)
m.sort_values("actual_aav_m", ascending=False)[
    ["player_name", "signing_year", "actual_aav_m", "mlbtr_aav_m", "model_aav_m"]].head(15).round(1)

MLBTR top 50 -> players with an actual contract the model can score:
  2024: 37 of 50
     not scored: Shohei Ohtani, Yoshinobu Yamamoto, Shota Imanaga, Jung Hoo Lee, Yariel Rodriguez, Mike Clevinger, Michael Lorenzen, Gio Urshela, Amed Rosario, Jason Heyward, Yuki Matsui, Brandon Belt, Jakob Junis
  2025: 44 of 50
     not scored: Hyeseong Kim, Nick Martinez, Jose Quintana, Tomoyuki Sugano, Shinnosuke Ogasawara, Spencer Turnbull
  2026: 39 of 50
     not scored: Munetaka Murakami, Tatsuya Imai, Zac Gallen, Brandon Woodruff, Trent Grisham, Kazuma Okamoto, Shota Imanaga, Lucas Giolito, Gleyber Torres, Cody Ponce, Max Scherzer

AAV error in $M (bias = predicted minus actual, negative = too low):
                 n  MLBTR MAE  Model MAE  MLBTR bias  Model bias
signing_year                                                    
2024          37.0       2.19       3.34        0.55       -1.71
2025          44.0       3.44       4.21       -1.56       -2.22
2026          39.0       3.72       4

,player_name,signing_year,actual_aav_m,mlbtr_aav_m,model_aav_m
100,Kyle Tucker,2026,60.0,36.4,33.1
50,Juan Soto,2025,51.0,46.2,39.0
101,Bo Bichette,2026,42.0,26.0,26.3
52,Alex Bregman,2025,40.0,26.0,24.0
105,Framber Valdez,2026,38.3,30.0,26.4
53,Blake Snell,2025,36.4,32.0,22.3
104,Alex Bregman,2026,35.0,26.7,20.0
51,Corbin Burnes,2025,35.0,28.6,28.1
107,Cody Bellinger,2026,32.5,28.0,20.7
3,Blake Snell,2024,31.0,28.6,26.8


In [9]:
import re, unicodedata
import pandas as pd

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = s.replace("-", " ")
    s = re.sub(r"\b(jr|sr|ii|iii|iv)\b\.?", "", s)
    s = re.sub(r"[^a-z ]", "", s)
    return re.sub(r"\s+", " ", s).strip()

# 1. Why are these domestic players unscored?
c = pd.read_csv("../data/processed/contracts.csv")
c["key"] = c["player_name"].apply(norm)
check = {
    2024: ["Mike Clevinger", "Michael Lorenzen", "Gio Urshela", "Amed Rosario",
           "Jason Heyward", "Brandon Belt", "Jakob Junis"],
    2025: ["Jose Quintana", "Spencer Turnbull"],
    2026: ["Lucas Giolito", "Max Scherzer"],
}
rows = []
for yr, names in check.items():
    for n in names:
        hit = c[(c["key"] == norm(n)) & (c["signing_year"] == yr)]
        rows.append({"player": n, "class": yr, "in_contracts_csv": len(hit) > 0,
                     "total_m": round(hit["total_value"].iloc[0] / 1e6, 1) if len(hit) else None,
                     "years": hit["years"].iloc[0] if len(hit) else None})
print(pd.DataFrame(rows).to_string(index=False))

# 2. How often was MLBTR closer, and does it depend on contract size?
m = pd.read_csv("../outputs/benchmark_mlbtr_vs_model.csv")
closer = m["mlbtr_err"].abs() < m["model_err"].abs()
print("\nMLBTR closer on %d of %d players (%.0f%%)" % (closer.sum(), len(m), 100 * closer.mean()))
m["size"] = pd.cut(m["actual_aav_m"], [0, 10, 20, 100], labels=["under $10M", "$10-20M", "$20M+"])
print(m.groupby("size", observed=True).agg(
    n=("mlbtr_err", "size"),
    mlbtr_mae=("mlbtr_err", lambda s: s.abs().mean()),
    model_mae=("model_err", lambda s: s.abs().mean()),
    mlbtr_bias=("mlbtr_err", "mean"),
    model_bias=("model_err", "mean")).round(2))

          player  class  in_contracts_csv  total_m  years
  Mike Clevinger   2024              True      3.0    1.0
Michael Lorenzen   2024              True      4.5    1.0
     Gio Urshela   2024             False      NaN    NaN
    Amed Rosario   2024              True      1.5    1.0
   Jason Heyward   2024             False      NaN    NaN
    Brandon Belt   2024             False      NaN    NaN
     Jakob Junis   2024             False      NaN    NaN
   Jose Quintana   2025              True      4.0    1.0
Spencer Turnbull   2025              True      1.3    1.0
   Lucas Giolito   2026              True      3.0    1.0
    Max Scherzer   2026              True      3.0    1.0

MLBTR closer on 73 of 120 players (61%)
             n  mlbtr_mae  model_mae  mlbtr_bias  model_bias
size                                                        
under $10M  21       2.36       2.60        1.34        2.20
$10-20M     69       2.58       2.84       -0.15       -1.43
$20M+       30     